# Matched-filter baseline for the BBH detector (same windows, same false-alarm calibration as the CNN)

**What this notebook does**

1. Builds a template bank (aligned-spin `IMRPhenomD`, quasi-random placement, `n_templates` waveforms) and a **per-example whitened matched filter**: every template is whitened with *the same PSD that whitened that example*, using the PSD table saved by the dataset notebook.
2. Searches every 2 s window over the whole bank and over all merger times allowed by the dataset (0.6–1.8 s), maximising over the signal phase (complex SNR).
3. Combines H1 and L1 into a **network statistic** `rho_net² = rho_H² + rho_L²`, with the L1 peak required within ±22 samples (≈ ±10.7 ms, the light-travel time between the detectors) of the H1 peak.
4. Adds the standard **Allen χ² consistency test** (16 frequency bands) and the *reweighted SNR* used by real pipelines. Plain SNR gets destroyed by glitches; the reweighted version is the fair baseline for a "robust to non-stationary noise" claim.
5. Compares CNN vs MF vs MF-reweighted: efficiency vs SNR at **equal false-alarm probability measured on the same `far` windows**, glitch robustness, and mass dependence.

**Honest limitations (put these in the write-up):** a finite aligned-spin bank (the injections are precessing half the time, so some loss is expected — the *fitting factor* diagnostic below measures it), no coherent two-detector sky-position search, no time-slide FAR calibration. Raise `n_templates` to tighten the baseline.

**How the noise normalisation works.** The data are 2 s windows cropped from whitened 4 s frames, so a long (low-mass) template is partly outside the window. Each `(template, lag)` statistic is therefore normalised empirically with glitch-free noise windows from `val` (mean noise `rho²` set to 2, as for a χ²₂ variable). That gives the optimal windowed-SNR normalisation without whitening every template per example.

## 1. Setup

In [1]:
import os, json, time, math, warnings
import numpy as np, pandas as pd, h5py
import matplotlib.pyplot as plt
from scipy import signal
from tqdm.auto import tqdm
import torch, torch.nn.functional as F

CFG = dict(
    data_dir="gw_bbh_dataset_v2",
    cnn_preds="runs/resnet1d_gn/predictions.npz",   # written by the training notebook (section 8) - set to your run folder
    run_dir="runs/matched_filter",
    # template bank
    n_templates=4000, bank_approximant="IMRPhenomD", bank_seed=11,
    bank_spin="effective",   # "effective": chi1z=chi2z (3-D bank, denser) | "independent": 4-D bank
    # how much data to search (the cost is linear in these). None = all.
    n_far=20000,             # noise-only windows for the false-alarm calibration (50 000 available in 'medium')
    n_bench_per_snr=500,     # signals per SNR bin (1000 available)
    n_cal=3000,              # glitch-free val noise windows used to normalise the statistic
    cal_clip=40.0,           # robust calibration: single-window rho^2 is clipped here (Gaussian noise never reaches it)
    cal_amp_cut=6.0,         # calibration windows with |whitened strain| > this many sigma are skipped
    # data-quality gating (standard in matched-filter pipelines): zero the data around huge whitened spikes before filtering
    gate_sigma=50.0, gate_pad_s=0.25, gate_taper_s=0.125,   # set gate_sigma=1e9 to switch gating off
    n_ff=300,                # bench signals (noise-free) used for the fitting-factor diagnostic
    # search
    batch=4, tchunk=250,     # lower both if you run out of GPU memory (3.7 GiB card: batch=2, tchunk=125)
    n_bands=16, coinc_samples=22,
)
os.makedirs(CFG["run_dir"], exist_ok=True); D = CFG["data_dir"]
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "(CPU - this will be very slow)")

/home/willowed/miniconda3/envs/gwml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda NVIDIA GeForce RTX 3050 Laptop GPU


## 2. Constants replicated from the dataset notebook (must match it exactly)

In [2]:
FS, WINDOW_S, PAD_S = 2048, 2.0, 1.0
N_WIN, N_FRAME, N_PAD = 4096, 8192, 2048
NB = N_FRAME // 2 + 1                                   # 4097 rfft bins
F_LOW, F_HIGH, F_LOWER_WF = 20.0, 1000.0, 15.0
STRAIN_SCALE = 1e21
T_PEAK_RANGE = (0.6, 1.8)                               # geocentric merger time inside the 2 s window [s]
FREQS = np.fft.rfftfreq(N_FRAME, 1.0 / FS); DF = FREQS[1] - FREQS[0]

def _band_mask(f=FREQS, lo=F_LOW, hi=F_HIGH, roll=3.0):
    m = np.zeros_like(f); m[(f >= lo) & (f <= hi)] = 1.0
    up = (f >= lo - roll) & (f < lo);  m[up] = 0.5 * (1 - np.cos(np.pi * (f[up] - (lo - roll)) / roll))
    dn = (f > hi) & (f <= hi + roll);  m[dn] = 0.5 * (1 + np.cos(np.pi * (f[dn] - hi) / roll))
    return m
BAND = _band_mask(); BW = float(np.sum(BAND ** 2) * DF); SNR2_FACTOR = 2.0 * BW / FS
FRAME_TAPER = signal.windows.tukey(N_FRAME, alpha=2 * PAD_S / (WINDOW_S + 2 * PAD_S))

def psd_to_whitener(psd):
    psd = np.asarray(psd, dtype=np.float64); G = np.zeros_like(psd)
    ok = (BAND > 0) & np.isfinite(psd) & (psd > 0)
    G[ok] = BAND[ok] / np.sqrt(psd[ok])
    return G / np.sqrt(BW)

def whiten_frames(frames, G):                           # used only by the self-test
    F_ = np.fft.rfft(frames * FRAME_TAPER, axis=-1)
    return np.fft.irfft(F_ * G, n=N_FRAME, axis=-1)[..., N_PAD:N_PAD + N_WIN]

# template merger sits at frame index N_FRAME//2; a lag s moves it to N_FRAME//2 + s. Allowed: merger inside 0.6-1.8 s of the window.
C_LAG = CFG["coinc_samples"]
S_LO = N_PAD + int(T_PEAK_RANGE[0] * FS) - N_FRAME // 2
S_HI = N_PAD + int(T_PEAK_RANGE[1] * FS) - N_FRAME // 2
S0 = S_HI - S_LO + 1                                    # lags that count
LAGS = np.arange(S_LO - C_LAG, S_HI + C_LAG + 1); NS = len(LAGS)   # + margin for the L1 coincidence window
print(f"SNR2_FACTOR = {SNR2_FACTOR:.4f} (BW = {BW:.2f} Hz) | {S0} merger lags + {C_LAG} margin each side")

SNR2_FACTOR = 0.9592 (BW = 982.25 Hz) | 2459 merger lags + 22 margin each side


## 3. The matched-filter engine

For one example and detector, with `x` the whitened window (zero-padded to 8192 samples) and `W = H_T·G` the template whitened with that example's `G`:

`z(s) = Σ_f X(f) · G(f) · conj(H_T(f)) · e^{2πifs/N}` (positive frequencies only → complex, so the phase is maximised for free).
The whitening filter moves from the template to the data (`X·G`), so the *same* template bank serves every PSD. The cost is one inverse FFT per (example, detector, template).

In [3]:
GATE_PAD, GATE_TAPER = int(CFG["gate_pad_s"] * FS), int(CFG["gate_taper_s"] * FS)

def gate(x, sigma, pad=GATE_PAD, taper=GATE_TAPER):
    """Zero the whitened data within +-pad of any |x| > sigma (smooth Hann edges), per detector. Returns (x_gated, was_gated (B,2))."""
    B, C, N = x.shape
    m = (x.abs() > sigma).float()
    flag = m.flatten(2).amax(-1) > 0
    if not bool(flag.any()): return x, flag
    core = F.max_pool1d(m, 2 * pad + 1, 1, pad)
    k = torch.hann_window(2 * taper + 1, periodic=False, device=x.device); k = (k / k.sum())[None, None]
    sm = F.conv1d(core.reshape(-1, 1, N), k, padding=taper).reshape(B, C, N)
    return x * (1 - sm).clamp(0, 1), flag

class MF:
    def __init__(self, H, tchunk=250, n_bands=16):
        self.Hc = H.conj().contiguous()                              # (T, 4097) conj of tapered template spectra
        self.Hpow = (H.real ** 2 + H.imag ** 2).contiguous()
        self.T, self.tchunk, self.p = H.shape[0], tchunk, n_bands
        self.cal = None                                              # {det: (T, NS)} noise normalisation
        self.lag_idx = torch.as_tensor(LAGS % N_FRAME, device=H.device)

    def prep(self, x, G):
        """x (B,2,4096) whitened windows, G (B,2,4097) whitening filters -> Y=rfft(pad(x))*G and template energies E (B,2,T)."""
        x, self.last_gated = gate(x, CFG["gate_sigma"])
        xp = F.pad(x, (N_PAD, N_FRAME - N_PAD - N_WIN))
        Y = torch.fft.rfft(xp, dim=-1) * G
        E = (2.0 / N_FRAME) * torch.matmul(G ** 2, self.Hpow.T)
        return Y, E.clamp_min(1e-30)

    def raw_rho2(self, Y, E, d, t0, t1):
        B = Y.shape[0]
        buf = torch.zeros(B, t1 - t0, N_FRAME, dtype=torch.complex64, device=Y.device)
        buf[..., 1:NB - 1] = Y[:, d, None, 1:NB - 1] * self.Hc[None, t0:t1, 1:NB - 1]
        w = torch.fft.ifft(buf, dim=-1) * 2.0
        r2 = (w.real ** 2 + w.imag ** 2)[..., self.lag_idx]
        return r2 / E[:, d, t0:t1, None]

    @torch.no_grad()
    def calibrate_accumulate(self, Y, E, acc, clip=40.0):
        # clipping keeps a few loud non-Gaussian windows from inflating the noise mean (which would shrink every calibrated SNR)
        for d in (0, 1):
            for t0 in range(0, self.T, self.tchunk):
                t1 = min(self.T, t0 + self.tchunk)
                acc[d][t0:t1] += self.raw_rho2(Y, E, d, t0, t1).clamp(max=clip).sum(0)

    def set_calibration(self, acc, n):
        self.cal = {d: (2.0 / (acc[d] / n).clamp_min(1e-9)).float() for d in (0, 1)}

    @torch.no_grad()
    def search(self, Y, E, G):
        B, dev = Y.shape[0], Y.device
        ar = torch.arange(B, device=dev)
        best = torch.full((B,), -1.0, device=dev)
        bT = torch.zeros(B, dtype=torch.long, device=dev); bsH = bT.clone(); bsL = bT.clone()
        bH = torch.zeros(B, device=dev); bL = bH.clone()
        offs = torch.arange(-C_LAG, C_LAG + 1, device=dev)
        for t0 in range(0, self.T, self.tchunk):
            t1 = min(self.T, t0 + self.tchunk)
            rH = self.raw_rho2(Y, E, 0, t0, t1) * self.cal[0][t0:t1][None]
            rL = self.raw_rho2(Y, E, 1, t0, t1) * self.cal[1][t0:t1][None]
            net = (rH + F.max_pool1d(rL, 2 * C_LAG + 1, 1, C_LAG))[..., C_LAG:C_LAG + S0]
            v, idx = net.flatten(1).max(1)
            tl, sH = idx // S0, idx % S0 + C_LAG
            gidx = (sH[:, None] + offs[None]).clamp(0, NS - 1)
            sL = gidx.gather(1, rL[ar, tl].gather(1, gidx).argmax(1, keepdim=True)).squeeze(1)
            upd = v > best
            best = torch.where(upd, v, best); bT = torch.where(upd, tl + t0, bT)
            bsH = torch.where(upd, sH, bsH); bsL = torch.where(upd, sL, bsL)
            bH = torch.where(upd, rH[ar, tl, sH], bH); bL = torch.where(upd, rL[ar, tl, sL], bL)
        lags = torch.as_tensor(LAGS, device=dev)
        chi = {}
        for d, s_idx in ((0, bsH), (1, bsL)):
            chi[d] = self.chi2_reduced(Y[:, d], self.Hc[bT], G[:, d], E[ar, d, bT], lags[s_idx], self.cal[d][bT, s_idx])
        return dict(rho2_net=best, T=bT, rho2_H=bH, rho2_L=bL, chi2r_H=chi[0], chi2r_L=chi[1], lag_H=lags[bsH], lag_L=lags[bsL])

    def chi2_reduced(self, Yd, Hc_sel, Gd, E_sel, lag, calf, return_zeta=False):
        """Allen chi^2 with p bands of equal template power, in the same (calibrated) units as rho."""
        p, B, dev = self.p, Yd.shape[0], Yd.device
        Zk = (Yd * Hc_sel)[:, 1:NB - 1]
        Pw = ((Hc_sel.real ** 2 + Hc_sel.imag ** 2) * Gd ** 2)[:, 1:NB - 1]
        kk = torch.arange(1, NB - 1, device=dev)
        ph = ((kk[None, :] * (lag[:, None] % N_FRAME)) % N_FRAME).float() * (2 * math.pi / N_FRAME)
        tr = Zk.real * torch.cos(ph) - Zk.imag * torch.sin(ph)
        ti = Zk.real * torch.sin(ph) + Zk.imag * torch.cos(ph)
        cs_r, cs_i = F.pad(tr.cumsum(1), (1, 0)), F.pad(ti.cumsum(1), (1, 0))
        cP = Pw.cumsum(1); cP = cP / cP[:, -1:].clamp_min(1e-30)
        tg = torch.linspace(0, 1, p + 1, device=dev)[1:-1][None].expand(B, -1).contiguous()
        inner = torch.searchsorted(cP.contiguous(), tg) + 1
        edges = torch.cat([torch.zeros(B, 1, dtype=torch.long, device=dev), inner.clamp(max=NB - 2),
                           torch.full((B, 1), NB - 2, dtype=torch.long, device=dev)], 1)
        sc = torch.sqrt(calf / E_sel)[:, None] * (2.0 / N_FRAME)
        zr = (cs_r.gather(1, edges[:, 1:]) - cs_r.gather(1, edges[:, :-1])) * sc
        zi = (cs_i.gather(1, edges[:, 1:]) - cs_i.gather(1, edges[:, :-1])) * sc
        Zr, Zi = zr.sum(1, keepdim=True), zi.sum(1, keepdim=True)
        chi2 = p * (((zr - Zr / p) ** 2) + ((zi - Zi / p) ** 2)).sum(1)
        out = chi2 / (2 * p - 2)
        return (out, Zr.squeeze(1) ** 2 + Zi.squeeze(1) ** 2) if return_zeta else out

def reweight(rho, chi2r):
    """Reweighted SNR (as in PyCBC): rho / [(1 + chi2r^3)/2]^(1/6) when chi2r > 1."""
    return rho / np.where(chi2r > 1, ((1 + chi2r ** 3) / 2) ** (1 / 6), 1.0)

## 4. Self-test on synthetic data (≈1 min on GPU)
Before touching the real dataset: inject known chirps into synthetic noise with a coloured PSD, whiten them **exactly as the dataset does**, and check that the engine (a) recovers the injected SNR, template, merger time and H1–L1 delay, (b) gives a χ² of about 1 for a clean signal and a large one for a glitch, and (c) the χ² code is consistent with the SNR code.

In [4]:
def newtonian_chirp(mc, n=N_FRAME, fmax=450.0):
    """Toy chirp with merger at frame index n//2 (self-test only; the real bank uses PyCBC waveforms)."""
    tau = (np.arange(n) - n // 2) / -FS                       # time before merger
    ok = tau > 1.0 / FS
    t = np.where(ok, tau, 1.0); Mt = mc * 4.925e-6
    f = (1 / np.pi) * (5 / (256 * t)) ** 0.375 * Mt ** -0.625
    ph = -2 * (5 * Mt) ** -0.625 * t ** 0.625
    h = np.where(ok & (f > 14) & (f < fmax), f ** (2 / 3) * np.cos(ph), 0.0)
    return h / np.sqrt(np.sum(h ** 2))

def selftest(T=12, B=4, seed=0):
    rng = np.random.default_rng(seed)
    mcs = np.linspace(8, 60, T)
    bank = np.stack([newtonian_chirp(m) for m in mcs]).astype(np.float32)
    H = torch.fft.rfft(torch.from_numpy(bank * FRAME_TAPER[None]).float(), dim=-1).to(DEV)
    mf = MF(H, tchunk=6, n_bands=CFG["n_bands"])
    psd = 1.0 + (30.0 / np.maximum(FREQS, 1.0)) ** 4 + (FREQS / 600.0) ** 2             # coloured toy PSD
    G = np.tile(psd_to_whitener(psd), (2, 1)).astype(np.float32)
    def noise(n):
        z = rng.standard_normal((n, 2, NB)) + 1j * rng.standard_normal((n, 2, NB))
        x = np.fft.irfft(z * BAND, n=N_WIN, axis=-1) if False else np.fft.irfft(z * BAND, n=N_FRAME, axis=-1)[..., :N_WIN]
        return (x / x.std()).astype(np.float32)
    Gt = torch.from_numpy(np.broadcast_to(G, (B, 2, NB)).copy()).to(DEV)
    # calibrate on noise
    acc = {d: torch.zeros(T, NS, device=DEV) for d in (0, 1)}; n_cal = 0
    for _ in range(25):
        x = torch.from_numpy(noise(B)).to(DEV); Y, E = mf.prep(x, Gt); mf.calibrate_accumulate(Y, E, acc); n_cal += B
    mf.set_calibration(acc, n_cal)
    # injections: template k, merger shifted by `shift`, H1/L1 phase-rotated, delay 6 samples, target in-window network SNR
    target, k, shift = 14.0, 7, 300
    rows = []
    for j in range(B):
        sig = []
        for d, (dly, rot) in enumerate([(0, 0.0), (6, 1.1)]):
            hh = np.roll(bank[k], shift + dly)
            an = np.fft.irfft(np.fft.rfft(hh) * np.where(FREQS > 0, np.exp(-1j * rot), 0) * 2, n=N_FRAME)   # phase-rotated real part
            sig.append(whiten_frames(an[None], G[d][None])[0])
        sig = np.stack(sig)
        w = np.sqrt(SNR2_FACTOR * (sig ** 2).sum(-1)); sig *= (target / np.sqrt((w ** 2).sum()))
        rows.append(sig)
    sigs = np.stack(rows).astype(np.float32)
    x = torch.from_numpy(sigs + noise(B)).to(DEV)
    Y, E = mf.prep(x, Gt); r = mf.search(Y, E, Gt)
    print("injected: template", k, "| lag", shift, "| H1->L1 delay 6 | network SNR", target)
    print("found   : template", r["T"].tolist(), "| lag_H", r["lag_H"].tolist(), "| lag_L-lag_H", (r["lag_L"] - r["lag_H"]).tolist())
    print("rho_net :", np.round(r["rho2_net"].sqrt().cpu().numpy(), 2), "| chi2r H/L:", np.round(r["chi2r_H"].cpu().numpy(), 2), np.round(r["chi2r_L"].cpu().numpy(), 2))
    # consistency: |zeta|^2 from the chi2 code must equal the calibrated rho^2 of the search
    ar = torch.arange(B, device=DEV); lags = torch.as_tensor(LAGS, device=DEV)
    _, z2 = mf.chi2_reduced(Y[:, 0], mf.Hc[r["T"]], Gt[:, 0], E[ar, 0, r["T"]], r["lag_H"], mf.cal[0][r["T"], r["lag_H"] - LAGS[0]], return_zeta=True)
    print("chi2-code |zeta|^2 vs search rho2_H: max rel diff", float(((z2 - r["rho2_H"]).abs() / r["rho2_H"]).max()))
    # glitch: a sine-gaussian of the same SNR, no chirp
    t = np.arange(N_WIN) / FS
    g = np.exp(-0.5 * ((t - 1.0) / 0.01) ** 2) * np.cos(2 * np.pi * 150 * (t - 1.0))
    gl = np.stack([g, g]).astype(np.float32); gl *= target / np.sqrt(SNR2_FACTOR * (gl ** 2).sum())
    xg = torch.from_numpy(np.broadcast_to(gl + noise(1), (B, 2, N_WIN)).copy()).to(DEV)
    Y, E = mf.prep(xg, Gt); rg = mf.search(Y, E, Gt)
    print("glitch  : rho_net", np.round(rg["rho2_net"].sqrt().cpu().numpy()[:2], 2), "| chi2r H/L", np.round(rg["chi2r_H"].cpu().numpy()[:2], 2), np.round(rg["chi2r_L"].cpu().numpy()[:2], 2),
          "-> reweighted", np.round(np.sqrt(reweight(rg["rho2_H"].sqrt().cpu().numpy(), rg["chi2r_H"].cpu().numpy()) ** 2 + reweight(rg["rho2_L"].sqrt().cpu().numpy(), rg["chi2r_L"].cpu().numpy()) ** 2)[:2], 2))
    # noise-only statistic: should be ~chi2_4-like and chi2r ~ 1
    vals, chis = [], []
    for _ in range(10):
        x = torch.from_numpy(noise(B)).to(DEV); Y, E = mf.prep(x, Gt); rn = mf.search(Y, E, Gt)
        vals.append(rn["rho2_net"].cpu().numpy()); chis.append(rn["chi2r_H"].cpu().numpy())
    print("noise   : median rho_net", round(float(np.sqrt(np.median(np.concatenate(vals)))), 2), "| median chi2r", round(float(np.median(np.concatenate(chis))), 2))

selftest()

injected: template 7 | lag 300 | H1->L1 delay 6 | network SNR 14.0
found   : template [7, 7, 7, 7] | lag_H [300, 301, 300, 300] | lag_L-lag_H [8, 4, 5, 6]
rho_net : [15.25 13.52 15.94 15.79] | chi2r H/L: [1.31 0.93 1.08 0.84] [1.2  1.27 1.11 1.2 ]
chi2-code |zeta|^2 vs search rho2_H: max rel diff 4.158002298026986e-07
glitch  : rho_net [5.7 5.7] | chi2r H/L [1.95 1.95] [1.48 1.48] -> reweighted [4.75 4.75]
noise   : median rho_net 5.21 | median chi2r 1.06


## 5. Template bank
Quasi-random (Sobol) placement over `m1∈[5,100]`, `m2∈[5,m1]` and the aligned spin (`bank_spin="effective"` uses `χ1z=χ2z=χ_eff∈[−0.9,0.9]`, which the phase evolution depends on to leading order, so the same number of templates covers the space much more densely than independent `χ1z,χ2z`). Half the templates are uniform in `Mc^{-5/3}` (dense at low mass, where signals are long and the bank needs the most templates) and half uniform in `ln Mc` (coverage up to total mass 200). Needs PyCBC (already installed for the dataset notebook). Waveforms are generated in a separate process that prints its progress; the finished bank is cached in the run folder.

In [5]:
import subprocess, sys
BANK_PATH = os.path.join(CFG["run_dir"], f"bank_{CFG['bank_approximant']}_{CFG['bank_spin']}_{CFG['n_templates']}.npz")
BANK_SCRIPT = os.path.join(CFG["run_dir"], "make_bank.py")

# The waveforms are generated in a SEPARATE clean Python process (no torch/CUDA, no Jupyter): forking worker processes from a
# kernel that already initialised CUDA can deadlock, which shows up as a progress bar stuck at 0 forever.
SCRIPT = r"""
import sys, os, json, time, numpy as np, pandas as pd
import multiprocessing as mp
cfg = json.loads(sys.argv[1])
FS, N_FRAME, STRAIN_SCALE, F_LOWER_WF = 2048, 8192, 1e21, 15.0

def sample_bank_params(n, seed):
    from scipy.stats import qmc
    sob = qmc.Sobol(d=5, scramble=True, seed=seed); out = []
    mc_lo, mc_hi = 4.35, 87.0
    while sum(len(o) for o in out) < n:
        u = sob.random(8192)
        x_lo, x_hi = mc_hi ** (-5 / 3), mc_lo ** (-5 / 3)
        mc = np.where(u[:, 4] < 0.5, (x_lo + (x_hi - x_lo) * u[:, 0]) ** (-3 / 5), np.exp(np.log(mc_lo) + (np.log(mc_hi) - np.log(mc_lo)) * u[:, 0]))
        eta_min = 5 * 100 / 105 ** 2
        eta = eta_min + (0.25 - eta_min) * u[:, 1]
        M = mc * eta ** (-0.6); sq = np.sqrt(np.maximum(1 - 4 * eta, 0))
        m1, m2 = 0.5 * M * (1 + sq), 0.5 * M * (1 - sq)
        ok = (m2 >= 5) & (m1 <= 100)
        out.append(pd.DataFrame(dict(mass1=m1, mass2=m2, spin1z=0.9 * (2 * u[:, 2] - 1), spin2z=0.9 * (2 * u[:, 3] - 1)))[ok])
    return pd.concat(out).iloc[:n].reset_index(drop=True)

def gen(p):
    from pycbc.waveform import get_td_waveform
    hp, hc = get_td_waveform(approximant=cfg["approx"], delta_t=1.0 / FS, f_lower=F_LOWER_WF, distance=1000.0,
                             inclination=0.0, coa_phase=0.0, **p)
    hp = hp.numpy() * STRAIN_SCALE; hc = hc.numpy() * STRAIN_SCALE
    ip = int(np.argmax(hp ** 2 + hc ** 2)); s0 = ip - N_FRAME // 2
    out = np.zeros(N_FRAME, dtype=np.float32); a, b = max(0, s0), min(len(hp), s0 + N_FRAME)
    out[a - s0:b - s0] = hp[a:b]
    return out / np.sqrt(np.sum(out.astype(np.float64) ** 2) + 1e-30)

def worker(p):
    try:
        return gen(p)
    except Exception as e:
        return None

if __name__ == "__main__":
    n = cfg["n"]; t0 = time.time()
    df = sample_bank_params(int(n * 1.05) + 8, cfg["seed"])
    if cfg["spin_eff"]: df["spin2z"] = df["spin1z"]          # 3-D bank (m1, m2, chi_eff): far denser per template than 4-D
    recs = [{k: float(r[k]) for k in df.columns} for _, r in df.iterrows()]
    # timing test on the most expensive (lowest-mass) waveform, in this process
    low = min(recs, key=lambda r: r["mass1"] + r["mass2"]); t1 = time.time(); gen(low)
    print(f"single low-mass waveform ({low['mass1']:.1f}+{low['mass2']:.1f} Msun): {time.time() - t1:.2f} s", flush=True)
    procs = max(1, min(cfg["procs"], 8)); print(f"generating {len(recs)} waveforms on {procs} processes", flush=True)
    res = []; ctx = mp.get_context("spawn")
    with ctx.Pool(procs) as pool:
        jobs = [pool.apply_async(worker, (r,)) for r in recs]
        for i in range(len(recs)):
            try:
                res.append(jobs[i].get(timeout=600))
            except mp.TimeoutError:
                print("TIMEOUT: no waveform finished for 10 minutes - aborting", flush=True); pool.terminate(); sys.exit(2)
            if (i + 1) % 25 == 0 or i + 1 == len(recs):
                el = time.time() - t0; print(f"  {i + 1}/{len(recs)} done | {el:.0f} s elapsed | ~{el / (i + 1) * (len(recs) - i - 1):.0f} s left", flush=True)
    ok = [i for i, r in enumerate(res) if r is not None][:n]
    if len(ok) < n:
        print(f"only {len(ok)}/{n} waveforms generated", flush=True); sys.exit(3)
    h = np.stack([res[i] for i in ok]).astype(np.float32); d = df.iloc[ok].reset_index(drop=True)
    np.savez_compressed(cfg["out"], h=h, params=d.values)
    print("saved", cfg["out"], h.shape, flush=True)
"""

if not os.path.exists(BANK_PATH):
    open(BANK_SCRIPT, "w").write(SCRIPT)
    args = json.dumps(dict(n=CFG["n_templates"], seed=CFG["bank_seed"], approx=CFG["bank_approximant"], spin_eff=(CFG["bank_spin"] == "effective"), out=BANK_PATH, procs=os.cpu_count() or 1))
    p = subprocess.Popen([sys.executable, BANK_SCRIPT, args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env=dict(os.environ, PYTHONWARNINGS="ignore"))
    for line in p.stdout:
        if "swiglal" not in line.lower() and line.strip(): print(line, end="")
    assert p.wait() == 0, "bank generation failed - see the messages above"
z = np.load(BANK_PATH); bank_h, bank_df = z["h"], pd.DataFrame(z["params"], columns=["mass1", "mass2", "spin1z", "spin2z"])
bank_df["mtotal"] = bank_df.mass1 + bank_df.mass2
print(bank_h.shape, "| total mass range", bank_df.mtotal.min().round(1), "-", bank_df.mtotal.max().round(1))
H_bank = torch.fft.rfft(torch.from_numpy(bank_h * FRAME_TAPER[None]).float(), dim=-1).to(DEV)
mf = MF(H_bank, tchunk=CFG["tchunk"], n_bands=CFG["n_bands"])

(4000, 8192) | total mass range 10.5 - 182.8


## 6. Data access: examples and their whitening filters

In [6]:
PSDF = h5py.File(os.path.join(D, "psd_table.h5"), "r"); _PSD, _G = {}, {}
def whitener_for(epoch, det, blk):
    key = (int(epoch), det, int(blk))
    if key not in _G:
        if (key[0], det) not in _PSD: _PSD[key[0], det] = PSDF[f"ep{key[0]:03d}/{det}"][:]
        _G[key] = psd_to_whitener(_PSD[key[0], det][key[2]]).astype(np.float32)
    return _G[key]

META = {s: pd.read_csv(os.path.join(D, f"{s}_meta.csv")) for s in ["val", "bench", "far"]}

def batch_G(meta_rows):
    return np.stack([np.stack([whitener_for(r.epoch_H1, "H1", r.blk_H1), whitener_for(r.epoch_L1, "L1", r.blk_L1)]) for r in meta_rows.itertuples()])

def iter_batches(split, rows, key="X", B=CFG["batch"]):
    with h5py.File(os.path.join(D, f"{split}.h5"), "r") as f:
        for a in range(0, len(rows), B):
            sel = rows[a:a + B]
            x = f[key][list(sel)] if len(sel) > 1 else f[key][int(sel[0]):int(sel[0]) + 1]
            yield sel, torch.from_numpy(np.asarray(x, dtype=np.float32)).to(DEV), torch.from_numpy(batch_G(META[split].iloc[sel])).to(DEV)

# row selections (the CNN is evaluated on exactly the same rows)
far_rows = np.arange(len(META["far"]) if CFG["n_far"] is None else min(CFG["n_far"], len(META["far"])))
bm_all = META["bench"]; snr_vals = np.sort(bm_all.snr_net.round(3).unique())
bench_rows = np.sort(np.concatenate([np.where(bm_all.snr_net.round(3) == s)[0][:CFG["n_bench_per_snr"] or None] for s in snr_vals]))
v = META["val"]; cal_rows = np.where((v.label == 0) & v.glitch_type.isna())[0][:CFG["n_cal"]]
ff_rows = bench_rows[np.linspace(0, len(bench_rows) - 1, min(CFG["n_ff"], len(bench_rows))).astype(int)]
print(f"far {len(far_rows)} | bench {len(bench_rows)} | calibration {len(cal_rows)} | fitting-factor {len(ff_rows)}")

far 20000 | bench 4500 | calibration 3000 | fitting-factor 300


## 7. Calibrate, then check the bank with noise-free signals

**Fitting factor** `FF = rho_MF(clean signal) / in-window optimal SNR`: how much SNR the bank loses (template mismatch from the finite bank, precession, higher modes). `FF = 1` is perfect; a real search bank targets ≥ 0.97. This number *is* the handicap of your baseline — report it.

In [7]:
t0 = time.time()
acc = {d: torch.zeros(mf.T, NS, device=DEV) for d in (0, 1)}
n_used = 0
for sel, x, G in tqdm(iter_batches("val", cal_rows), total=math.ceil(len(cal_rows) / CFG["batch"]), desc="calibration"):
    keep = x.abs().amax((1, 2)) < CFG["cal_amp_cut"]                 # skip windows with loud transients
    if not keep.any(): continue
    x, G = x[keep], G[keep]
    Y, E = mf.prep(x, G); mf.calibrate_accumulate(Y, E, acc, clip=CFG["cal_clip"]); n_used += len(x)
mf.set_calibration(acc, n_used)
print(f"calibration used {n_used}/{len(cal_rows)} windows ({len(cal_rows) - n_used} skipped for loud transients)")
print(f"calibration done in {time.time() - t0:.0f} s  ({(time.time() - t0) / len(cal_rows) * 1e3:.0f} ms / window)  -> expected full run ~ "
      f"{(len(far_rows) + len(bench_rows)) * (time.time() - t0) / len(cal_rows) / 60:.0f} min")

def run_search(split, rows, key="X", desc=""):
    out = {k: [] for k in ["rho2_net", "T", "rho2_H", "rho2_L", "chi2r_H", "chi2r_L", "lag_H", "lag_L", "xmax", "gated"]}
    for sel, x, G in tqdm(iter_batches(split, rows, key), total=math.ceil(len(rows) / CFG["batch"]), desc=desc):
        out["xmax"].append(x.abs().amax((1, 2)).cpu().numpy())                 # largest whitened sample before gating
        Y, E = mf.prep(x, G); r = mf.search(Y, E, G)
        out["gated"].append(mf.last_gated.any(1).cpu().numpy())
        for k in r: out[k].append(r[k].cpu().numpy())
    out = {k: np.concatenate(v) for k, v in out.items()}
    out["rho_net"] = np.sqrt(out["rho2_net"])
    out["rho_rw"] = np.sqrt(reweight(np.sqrt(out["rho2_H"]), out["chi2r_H"]) ** 2 + reweight(np.sqrt(out["rho2_L"]), out["chi2r_L"]) ** 2)
    return out

# in-window optimal SNR of every bench signal (from the stored clean signal S)
with h5py.File(os.path.join(D, "bench.h5"), "r") as f:
    assert "S" in f, "bench.h5 has no clean-signal array S"
    snr_win_all = np.concatenate([np.sqrt(SNR2_FACTOR * (f["S"][a:a + 500].astype(np.float32) ** 2).sum((-1, -2))) for a in range(0, len(f["S"]), 500)])
bm_all["snr_win"] = snr_win_all

ff = run_search("bench", ff_rows, key="S", desc="fitting factor (noise-free)")
FF = pd.DataFrame(dict(ff=ff["rho_net"] / snr_win_all[ff_rows], mtotal=bm_all.mtotal.values[ff_rows], chi_p=bm_all.chi_p.values[ff_rows],
                       aligned=bm_all.aligned_spins.values[ff_rows]))
print(f"fitting factor: median {FF.ff.median():.3f} | 10th percentile {FF.ff.quantile(.1):.3f}")
display(FF.groupby(pd.cut(FF.mtotal, [10, 25, 40, 60, 100, 210]), observed=True).ff.agg(["median", "count"]).round(3).T)
display(FF.groupby("aligned").ff.agg(["median", "count"]).round(3).T)

calibration:   0%|          | 0/750 [00:00<?, ?it/s][W1013 07:26:27.538210096 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 67108864 bytes (free: 7798784, total: 3951296512).
[W1013 07:26:27.542187523 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 67108864 bytes (free: 45547520, total: 3951296512).
calibration:   0%|          | 0/750 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacity of 3.68 GiB of which 43.44 MiB is free. Process 75329 has 492.00 MiB memory in use. Process 158973 has 896.00 MiB memory in use. Process 161896 has 896.00 MiB memory in use. Process 164545 has 896.00 MiB memory in use. Including non-PyTorch memory, this process has 502.00 MiB memory in use. Of the allocated memory 336.42 MiB is allocated by PyTorch, and 67.58 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

**Reading the fitting factor:** the high-mass bins should be near 1. If low-mass FF is poor, raise `n_templates` (2000–4000). If *everything* is far below ~0.8 (say median < 0.7), something does not match the dataset (the self-test above guards the engine, so look at `psd_table.h5` indexing and the constants in section 2). Precessing injections (`aligned=False`) will show a lower FF than aligned ones — that is an honest limitation of an aligned-spin bank.

## 8. Run the search on the noise-only (`far`) and signal (`bench`) windows

In [ ]:
RES_FAR, RES_BENCH = [os.path.join(CFG["run_dir"], f"mf_{s}.npz") for s in ("far", "bench")]
def _cache_ok():
    if not (os.path.exists(RES_FAR) and os.path.exists(RES_BENCH)): return False
    z = np.load(RES_FAR)
    return ("bank_id" in z.files and str(z["bank_id"]) == BANK_ID and z["rows"].tolist() == far_rows.tolist()
            and np.load(RES_BENCH)["rows"].tolist() == bench_rows.tolist())
# results are reused only for the SAME bank, the SAME calibration and the same rows (a changed calibration must trigger a new search)
BANK_ID = f"{os.path.basename(BANK_PATH)}|cal={float(mf.cal[0].sum() + mf.cal[1].sum()):.3e}|clip={CFG['cal_clip']}|amp={CFG['cal_amp_cut']}|gate={CFG['gate_sigma']}/{CFG['gate_pad_s']}/{CFG['gate_taper_s']}"
if _cache_ok():
    mf_far, mf_bench = dict(np.load(RES_FAR)), dict(np.load(RES_BENCH)); print("loaded cached results for", BANK_ID)
else:
    mf_far = run_search("far", far_rows, desc="far (noise only)");      mf_far["rows"] = far_rows; mf_far["bank_id"] = np.array(BANK_ID)
    mf_bench = run_search("bench", bench_rows, desc="bench (signals)"); mf_bench["rows"] = bench_rows; mf_bench["bank_id"] = np.array(BANK_ID)
    np.savez_compressed(RES_FAR, **mf_far); np.savez_compressed(RES_BENCH, **mf_bench)
print({k: v.shape for k, v in mf_far.items()})

## 9. Comparison with the CNN at equal false-alarm probability
Thresholds for **every** method come from the same noise-only `far` windows: the score exceeded by a fraction `p` of them. Efficiency = fraction of `bench` signals above that threshold.

In [ ]:
P = np.load(CFG["cnn_preds"])
far_scores = {"CNN": P["far_logit"][far_rows], "MF (SNR)": mf_far["rho_net"], "MF (reweighted SNR)": mf_far["rho_rw"]}
bench_scores = {"CNN": P["bench_logit"][bench_rows], "MF (SNR)": mf_bench["rho_net"], "MF (reweighted SNR)": mf_bench["rho_rw"]}
def thr_at(s, p): s = np.sort(s)[::-1]; return float(s[min(int(p * len(s)), len(s) - 1)])
FAPS = [f for f in (1e-2, 1e-3, 1e-4) if f * len(far_rows) >= 10]
THR = {m: {f: thr_at(far_scores[m], f) for f in FAPS} for m in far_scores}
bm = bm_all.iloc[bench_rows].copy().reset_index(drop=True); bm["snr_bin"] = bm.snr_net.round(3)
for m in bench_scores: bm[m] = bench_scores[m]
assert FAPS, f"only {len(far_rows)} noise windows: need at least 1000 for FAP 1e-2 (use more far windows)"
print("false-alarm resolution:", len(far_rows), "windows ->", FAPS)

rows = []
for s, g in bm.groupby("snr_bin"):
    r = {"snr": s, "n": len(g)}
    for f in FAPS:
        for m in far_scores: r[f"{m} @{f:g}"] = (g[m] > THR[m][f]).mean()
    rows.append(r)
EFF = pd.DataFrame(rows); display(EFF.round(3)); EFF.to_csv(os.path.join(CFG["run_dir"], "efficiency_vs_snr_all_methods.csv"), index=False)

fig, ax = plt.subplots(1, len(FAPS), figsize=(6 * len(FAPS), 4.4), squeeze=False)
for a, f in zip(ax[0], FAPS):
    for m, st in zip(far_scores, ["-o", "--s", "-.^"]):
        p = EFF[f"{m} @{f:g}"].values; a.errorbar(EFF.snr, p, yerr=np.sqrt(p * (1 - p) / EFF.n.values), fmt=st, capsize=2, label=m)
    a.set(xlabel="labelled network SNR (full 4 s frame)", ylabel="detection efficiency", title=f"FAP = {f:g} per window", ylim=(-.02, 1.02)); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig(os.path.join(CFG["run_dir"], "cnn_vs_mf_efficiency.png"), dpi=130); plt.show()

### 9.1 Efficiency vs the SNR actually inside the 2 s window (the fair axis for long, low-mass signals)

In [ ]:
edges = [0, 5, 6, 7, 8, 9, 10, 12, 15, 20, 30, 1e3]
bm["wbin"] = pd.cut(bm.snr_win, edges)
fref = 1e-3 if 1e-3 in FAPS else FAPS[-1]
rows = []
for b, g in bm.groupby("wbin", observed=True):
    r = {"in-window SNR": str(b), "n": len(g)}
    for m in far_scores: r[m] = (g[m] > THR[m][fref]).mean()
    rows.append(r)
EW = pd.DataFrame(rows); display(EW.round(3))
fig, ax = plt.subplots(figsize=(7, 4.4)); xs = [(b.left + min(b.right, 40)) / 2 for b in bm.groupby("wbin", observed=True).size().index]
for m, st in zip(far_scores, ["-o", "--s", "-.^"]): ax.plot(xs, EW[m], st, label=m)
ax.set(xlabel="in-window optimal network SNR", ylabel="efficiency", title=f"efficiency at FAP = {fref:g}", ylim=(-.02, 1.02)); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(CFG["run_dir"], "cnn_vs_mf_window_snr.png"), dpi=130); plt.show()

### 9.2 Mass dependence (in-window SNR 8–15) and glitch robustness

In [ ]:
sel = bm[(bm.snr_win >= 8) & (bm.snr_win <= 15)].copy()
sel["mbin"] = pd.cut(sel.mtotal, [10, 25, 40, 60, 80, 110, 150, 210])
tab = sel.groupby("mbin", observed=True).apply(lambda g: pd.Series({**{m: (g[m] > THR[m][fref]).mean() for m in far_scores}, "n": len(g)}), include_groups=False)
print(f"efficiency by total mass, in-window SNR 8-15, FAP {fref:g}"); display(tab.round(3))

fm = META["far"].iloc[far_rows].reset_index(drop=True); gl = fm.glitch_type.notna().values
rows = []
for m in far_scores:
    for f in FAPS:
        ab = far_scores[m] > THR[m][f]
        r = {"method": m, "FAP": f, "clean noise": ab[~gl].mean(), "glitchy noise": ab[gl].mean(), "glitchy / clean": ab[gl].mean() / max(ab[~gl].mean(), 1e-9)}
        for k in sorted(fm.glitch_type.dropna().unique()): r[k] = ab[(fm.glitch_type == k).values].mean()
        rows.append(r)
print("false-alarm probability by noise type (equal overall FAP):"); display(pd.DataFrame(rows).round(4))
bm["glitch"] = bm.glitch_type.notna()
g8 = bm[(bm.snr_win >= 8) & (bm.snr_win <= 15)]
display(g8.groupby("glitch").apply(lambda g: pd.Series({m: (g[m] > THR[m][fref]).mean() for m in far_scores} | {"n": len(g)}), include_groups=False)
        .rename(index={False: "signal, clean noise", True: "signal + glitch"}).round(3))

### 9.3 What drives the matched-filter false alarms? (run this before interpreting the MF curves)

In [ ]:
fm = META["far"].iloc[far_rows].reset_index(drop=True)
d = pd.DataFrame(dict(rho=mf_far["rho_net"], rho_rw=mf_far["rho_rw"], chi2r_H=mf_far["chi2r_H"], chi2r_L=mf_far["chi2r_L"],
                      template_mtotal=bank_df.mtotal.values[mf_far["T"]], lag_H=mf_far["lag_H"],
                      glitch=fm.glitch_type.fillna("none").values, cnn_logit=far_scores["CNN"]))
q = [.5, .9, .99, .999]
print("MF network SNR in noise-only windows (quantiles):")
display(pd.DataFrame({"all": d.rho.quantile(q), "clean only": d[d.glitch == "none"].rho.quantile(q), "glitchy only": d[d.glitch != "none"].rho.quantile(q)}).round(2).T)
print("20 loudest noise windows for the plain MF (is a clean window, a glitch, or a window edge responsible?):")
display(d.sort_values("rho", ascending=False).head(20).round(2))
print("merger-lag of the best template for the top 1% of windows vs all (lags near the window edges = short, burst-like templates):")
top = d.rho >= d.rho.quantile(.99)
display(pd.DataFrame({"top 1%": d[top].lag_H.describe(), "all": d.lag_H.describe()}).round(0).T)

### 9.4 Inspect the loudest noise windows: transient, spectral line, or something else?

In [ ]:
Nf = np.fft.rfftfreq(N_WIN, 1.0 / FS); inb = (Nf > 25) & (Nf < 900)
def smooth(p, k=16): return np.convolve(p, np.ones(k) / k, "same")
top_idx = d.sort_values("rho", ascending=False).head(40).index
rows = []
with h5py.File(os.path.join(D, "far.h5"), "r") as f:
    for idx in top_idx:
        x = f["X"][int(far_rows[idx])].astype(np.float32)
        P = np.abs(np.fft.rfft(x, axis=-1)) ** 2; Ps = np.stack([smooth(p) for p in P])
        rows.append(dict(idx=idx, rho=d.rho[idx], rho_rw=d.rho_rw[idx], glitch=d.glitch[idx],
                         xmax_H1=np.abs(x[0]).max(), xmax_L1=np.abs(x[1]).max(),
                         line_excess_H1=Ps[0][inb].max() / np.median(Ps[0][inb]), line_excess_L1=Ps[1][inb].max() / np.median(Ps[1][inb])))
T40 = pd.DataFrame(rows).round(2); display(T40.head(20))
print("Gaussian expectation: xmax ~ 4-4.5 sigma, line_excess < ~1.5 (ratio of the highest smoothed spectral bin to the median)")
print("Windows with xmax > 8 (time-domain transient):", int(((T40.xmax_H1 > 8) | (T40.xmax_L1 > 8)).sum()),
      "| with line_excess > 3 (narrow-band excess):", int(((T40.line_excess_H1 > 3) | (T40.line_excess_L1 > 3)).sum()), "of", len(T40))
fig, ax = plt.subplots(4, 2, figsize=(14, 9))
with h5py.File(os.path.join(D, "far.h5"), "r") as f:
    for i, idx in enumerate(top_idx[:4]):
        x = f["X"][int(far_rows[idx])].astype(np.float32)
        ax[i, 0].plot(np.arange(N_WIN) / FS, x[0], lw=.5, label="H1"); ax[i, 0].plot(np.arange(N_WIN) / FS, x[1], lw=.5, label="L1")
        ax[i, 0].set(title=f"row {idx}: rho={d.rho[idx]:.0f}, rho_rw={d.rho_rw[idx]:.1f}, glitch={d.glitch[idx]}"); ax[i, 0].legend(loc="upper right")
        for k, nm in enumerate(["H1", "L1"]): ax[i, 1].semilogy(Nf, smooth(np.abs(np.fft.rfft(x[k])) ** 2), lw=.7, label=nm)
        ax[i, 1].set(xlim=(0, 1024), title="smoothed spectrum of the whitened window (flat = well whitened)")
plt.tight_layout(); plt.savefig(os.path.join(CFG["run_dir"], "loudest_noise_windows.png"), dpi=120); plt.show()

### 9.5 Same comparison with an explicit data-quality cut (applied identically to the CNN and the matched filter)
Real strain contains rare huge transients (hundreds to thousands of sigma in the whitened data). Matched-filter pipelines remove them with gating or data-quality vetoes. Here the **same** windows are removed for *all* methods, and thresholds are recomputed from the remaining noise windows — a result that does not depend on how the matched filter handles those spikes.

In [ ]:
ok_far, ok_b = mf_far["xmax"] < CFG["gate_sigma"], mf_bench["xmax"] < CFG["gate_sigma"]
print(f"windows containing a > {CFG['gate_sigma']:.0f} sigma spike: far {100 * (~ok_far).mean():.2f}% ({int((~ok_far).sum())}) | bench {100 * (~ok_b).mean():.2f}% | "
      f"windows gated in the MF: far {int(mf_far['gated'].sum())}, bench {int(mf_bench['gated'].sum())}")
FAPS2 = [f for f in (1e-2, 1e-3, 1e-4) if f * ok_far.sum() >= 10]; fref2 = 1e-3 if 1e-3 in FAPS2 else FAPS2[-1]
THR2 = {m: {f: thr_at(far_scores[m][ok_far], f) for f in FAPS2} for m in far_scores}
bq = bm[ok_b]; rows = []
for b, g in bq.groupby("wbin", observed=True):
    r = {"in-window SNR": str(b), "n": len(g)}
    for m in far_scores: r[m] = (g[m] > THR2[m][fref2]).mean()
    rows.append(r)
EQ = pd.DataFrame(rows); print(f"efficiency at FAP = {fref2:g}, spike windows removed everywhere"); display(EQ.round(3))
fig, ax = plt.subplots(figsize=(7, 4.4)); xs = [(b.left + min(b.right, 40)) / 2 for b in bq.groupby("wbin", observed=True).size().index]
for m, st in zip(far_scores, ["-o", "--s", "-.^"]): ax.plot(xs, EQ[m], st, label=m)
ax.set(xlabel="in-window optimal network SNR", ylabel="efficiency", title=f"data-quality cut, FAP = {fref2:g}", ylim=(-.02, 1.02)); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(CFG["run_dir"], "cnn_vs_mf_dq_cut.png"), dpi=130); plt.show()
fm2 = META["far"].iloc[far_rows].reset_index(drop=True)[ok_far]; gl2 = fm2.glitch_type.notna().values; rows = []
for m in far_scores:
    ab = far_scores[m][ok_far] > THR2[m][fref2]
    r = {"method": m, "clean noise": ab[~gl2].mean(), "glitchy noise": ab[gl2].mean()}
    for k in sorted(fm2.glitch_type.dropna().unique()): r[k] = ab[(fm2.glitch_type == k).values].mean()
    rows.append(r)
print(f"false-alarm probability by noise type at FAP = {fref2:g} (spike windows removed)"); display(pd.DataFrame(rows).round(4))

## 10. Save a summary

In [ ]:
summary = dict(n_templates=int(mf.T), fitting_factor_median=float(FF.ff.median()), fitting_factor_p10=float(FF.ff.quantile(.1)),
               n_far=int(len(far_rows)), n_bench=int(len(bench_rows)), faps=FAPS,
               thresholds={m: {str(f): t for f, t in THR[m].items()} for m in THR},
               efficiency_vs_snr=EFF.round(4).to_dict(orient="list"), config=CFG)
json.dump(summary, open(os.path.join(CFG["run_dir"], "mf_summary.json"), "w"), indent=1, default=float)
print("saved:", sorted(os.listdir(CFG["run_dir"])))

## 11. Interpreting the result (for your write-up)

* Report the **fitting factor** next to the MF curve — it states how strong your baseline is.
* The comparison is only as good as the false-alarm calibration: with `n_far = 20 000` windows, FAP = 1e-3 rests on ~20 events (≈ ±22 %). Use all 50 000 (or more `far` windows) for headline numbers.
* Plain SNR vs reweighted SNR shows what a signal-consistency test buys against glitches; the CNN should be compared with the **reweighted** one.
* Natural extensions: a denser bank (precessing or higher-mode templates), a coherent two-detector statistic, and calibrating FAR in physical units with time slides.